# 05 — Bit Manipulation

## Why This Matters for DSA
Bit tricks solve a narrow but real category of problems in genuinely optimal time and space — often O(n) time and O(1) space where the "obvious" approach costs O(n) space (a hash set) or O(n log n) time (sorting). They also show up as a representation technique, not just an optimization: encoding a subset as an integer (Section 6) is the foundation of bitmask dynamic programming, a technique you'll meet again later in this course. This notebook is deliberately narrower than most — bit tricks are a toolbox of specific, nameable techniques rather than one big generalizable idea, so it reads more like a reference than earlier notebooks.

## Prerequisites
- `03_Backtracking` (Phase 01) — Section 7's discussion of subset generation, directly connected to this notebook's bitmask subset section
- Basic familiarity with binary representation (this notebook briefly recaps it, but doesn't teach binary from zero)

## Learning Objectives
By the end of this notebook, you will be able to:
- Explain what each bitwise operator (`&`, `|`, `^`, `~`, `<<`, `>>`) actually does at the bit level, and why each is the natural choice for its common use case
- Check, set, clear, and toggle an individual bit, and explain why each operation uses the specific operator it does
- Apply the XOR-cancellation trick to find a single non-duplicate element in O(n) time and O(1) space
- Apply Brian Kernighan's algorithm to count set bits proportional to the number of SET bits, not the total bit width
- Check whether a number is a power of two, and isolate a number's lowest set bit, using the same underlying `n & (n-1)` family of tricks
- Represent and enumerate all subsets of a set using integer bitmasks, as an iterative alternative to backtracking's recursive include/exclude
- Judge when a bit trick is worth reaching for versus when it trades away code clarity for a marginal or nonexistent practical gain

## Checklist Before Moving On
- [ ] I can state, without hesitation, which operator to use for checking/setting/clearing/toggling a bit, and why
- [ ] I can explain why XOR-ing an array where every element appears twice except one isolates that one element
- [ ] I can trace `n & (n-1)` by hand and explain why it clears exactly the lowest set bit
- [ ] I can explain why `n & (n-1) == 0` (with `n > 0`) correctly identifies powers of two
- [ ] I can generate all subsets of a set using bitmasks and connect it back to `03_Backtracking`'s recursive version
- [ ] I know at least one situation where reaching for a bit trick would hurt more than help

## Table of Contents
1. Bitwise Operators Recap — AND, OR, XOR, NOT, Shifts
2. Check, Set, Clear, and Toggle a Bit
3. The Power of XOR — Finding the Single Non-Duplicate
4. Counting Set Bits — Brian Kernighan's Algorithm
5. Power of Two and Isolating the Lowest Set Bit
6. Bitmasking for Subsets
7. When Bit Tricks Are (and Aren't) Worth It
8. Phase Checkpoint, Cheat Sheet, and Answer Key
9. LeetCode Practice Problems


## 1. Bitwise Operators Recap — AND, OR, XOR, NOT, Shifts

### The Why
Everything in this notebook is built from six operators. Being fluent enough to predict their output without running code is the actual prerequisite skill — the rest of the notebook is really just "clever combinations of these six things," so it's worth pausing here even if the operators themselves feel familiar.

### Core Mechanism
- **AND (`&`):** result bit is 1 only where BOTH operand bits are 1. Used for masking — isolating specific bits while zeroing out everything else.
- **OR (`|`):** result bit is 1 where EITHER operand bit is 1. Used for setting bits — forcing specific bits on without disturbing others.
- **XOR (`^`):** result bit is 1 where the operand bits DIFFER. Used for toggling bits, and has a special algebraic property this notebook leans on repeatedly: `x ^ x == 0` and `x ^ 0 == x`.
- **NOT (`~`):** flips every single bit of the operand's underlying representation — including the sign bit for a signed integer type, which is why `~n` is not simply "n with its visible bits flipped," it flips ALL bits of however many the type actually has.
- **Left shift (`<<`):** shifts bits left, filling with 0s on the right — equivalent to multiplying by 2 per shift position, as long as no overflow occurs.
- **Right shift (`>>`):** shifts bits right — equivalent to dividing by 2 per shift position. For signed negative numbers, right shift is now guaranteed (as of C++20) to be an arithmetic shift (sign-extending); on earlier standards this was implementation-defined, worth knowing if working with legacy code or strict portability requirements.

### Common Pitfalls
- Assuming `~n` only flips the bits you can visually see printed — it flips every bit of the full underlying representation (commonly 32 bits for `int`), which is why `~n` for a positive `n` produces what looks like an unexpectedly large negative number rather than a small one.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <bitset>
using namespace std;

int main() {
    int a = 12;   // binary: 1100
    int b = 10;   // binary: 1010

    // AND (&): 1 only where BOTH bits are 1 -- used for masking (isolating specific bits)
    cout << "a & b = " << bitset<8>(a & b) << " (" << (a & b) << ")\n";   // 1000 (8)

    // OR (|): 1 where EITHER bit is 1 -- used for setting bits
    cout << "a | b = " << bitset<8>(a | b) << " (" << (a | b) << ")\n";   // 1110 (14)

    // XOR (^): 1 where the bits DIFFER -- used for toggling bits, and has the special
    // property that x ^ x == 0 and x ^ 0 == x (Section 3 builds directly on this)
    cout << "a ^ b = " << bitset<8>(a ^ b) << " (" << (a ^ b) << ")\n";   // 0110 (6)

    // NOT (~): flips every bit -- on a signed int this includes the sign bit, so ~a is
    // NOT simply "the complement within a's apparent bit width" -- it flips ALL 32 (or
    // however many) bits of the underlying representation
    cout << "~a = " << (~a) << " (flips all bits, including ones outside what's printed above)\n";

    // LEFT SHIFT (<<): shifts bits left, filling with 0s on the right -- equivalent to
    // multiplying by 2 per shift (as long as no overflow occurs)
    cout << "1 << 4 = " << (1 << 4) << " (multiply 1 by 2^4 = 16)\n";

    // RIGHT SHIFT (>>): shifts bits right -- equivalent to dividing by 2 per shift,
    // rounding toward negative infinity for signed negative numbers (implementation-defined
    // before C++20, arithmetic shift is now guaranteed behavior as of C++20)
    cout << "20 >> 2 = " << (20 >> 2) << " (divide 20 by 2^2 = 5)\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Check, Set, Clear, and Toggle a Bit

### The Why
These four operations are the actual building blocks nearly every other bit trick in this notebook (and in the wild) is assembled from. Knowing WHY each one uses its specific operator — not just memorizing the four formulas — is what lets you derive a fifth one confidently if a problem needs something slightly different.

### Core Mechanism
- **Check bit `i`:** `(n >> i) & 1` — shift the target bit down to position 0, then mask with 1 to isolate just that bit.
- **Set bit `i`:** `n | (1 << i)` — OR-ing with 0 leaves a bit unchanged, OR-ing with 1 forces it to 1. Since `1 << i` is 0 everywhere except position `i`, this forces bit `i` on while leaving every other bit exactly as it was.
- **Clear bit `i`:** `n & ~(1 << i)` — AND-ing with 1 leaves a bit unchanged, AND-ing with 0 forces it to 0. `~(1 << i)` is 1 everywhere except position `i` (where it's 0), so this forces bit `i` off while preserving everything else.
- **Toggle bit `i`:** `n ^ (1 << i)` — XOR-ing with 0 leaves a bit unchanged, XOR-ing with 1 flips it. Same targeted-position logic as the other two.
- **The underlying pattern:** each operator was chosen because its "identity" behavior (the input value that leaves an operand bit completely unchanged) is exactly what's needed for every OTHER bit position besides the one being targeted. This is the actual insight worth internalizing, not the four formulas themselves.

### Common Pitfalls
- Using OR when you meant to clear a bit, or AND when you meant to set one — a quick sanity check: OR can only ever turn bits ON (never off), AND can only ever turn bits OFF (never on) relative to a bare `1 << i` mask; if a formula needs to *force* a bit to a specific state, match the operator to which direction it forces.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <bitset>
using namespace std;

// FOUR OPERATIONS EVERY BIT-MANIPULATION PROBLEM REUSES: check, set, clear, toggle a
// specific bit at position i (0-indexed from the right, the least significant bit).

bool checkBit(int n, int i) {
    return (n >> i) & 1;   // shift the target bit down to position 0, then mask with 1
}

int setBit(int n, int i) {
    return n | (1 << i);   // OR with a 1 at position i -- forces that bit on, leaves others untouched
}

int clearBit(int n, int i) {
    return n & ~(1 << i);   // AND with all-1s EXCEPT position i -- forces that bit off,
                             // leaves others untouched (the ~ flips (1<<i) to all-1s-but-that-bit)
}

int toggleBit(int n, int i) {
    return n ^ (1 << i);   // XOR with a 1 at position i -- flips that bit, leaves others untouched
}

int main() {
    int n = 0b1010;   // 10 in decimal

    cout << "n = " << bitset<8>(n) << "\n";
    cout << "checkBit(n, 1) = " << checkBit(n, 1) << " (bit 1 is set)\n";
    cout << "checkBit(n, 0) = " << checkBit(n, 0) << " (bit 0 is not set)\n";

    cout << "setBit(n, 0)   = " << bitset<8>(setBit(n, 0)) << "\n";
    cout << "clearBit(n, 1) = " << bitset<8>(clearBit(n, 1)) << "\n";
    cout << "toggleBit(n, 3)= " << bitset<8>(toggleBit(n, 3)) << "\n";

    // WHY EACH OPERATION USES ITS SPECIFIC OPERATOR, NOT A DIFFERENT ONE:
    // - check uses AND with 1: isolates exactly one bit, everything else becomes 0
    // - set uses OR: OR-ing with 0 leaves a bit unchanged, OR-ing with 1 forces it to 1 --
    //   exactly "force on, don't touch anything else"
    // - clear uses AND with NOT: AND-ing with 1 leaves a bit unchanged, AND-ing with 0
    //   forces it to 0 -- exactly "force off, don't touch anything else"
    // - toggle uses XOR: XOR-ing with 0 leaves a bit unchanged, XOR-ing with 1 flips it --
    //   exactly "flip this one, don't touch anything else"
    // each operator was chosen because its "identity" behavior (leaves bit unchanged) matches
    // exactly what's needed for all the OTHER bit positions

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. The Power of XOR — Finding the Single Non-Duplicate

### The Why
This is the single most quoted "clever" bit trick in interview prep, and it's worth actually understanding rather than memorizing, because the underlying reasoning (cancellation via XOR's algebraic properties) generalizes to other XOR-based problems you'll meet later.

### Core Mechanism
- Given an array where every element appears exactly twice EXCEPT one element that appears once, find that element.
- **The properties that make this work:** `x ^ x == 0` (a value XORed with itself cancels to zero), `x ^ 0 == x` (XOR with zero is the identity), and XOR is both commutative and associative (order and grouping don't matter for the final result).
- XOR every element in the array together. Every DUPLICATED value contributes a pair that cancels to 0 (in whatever order they appear, thanks to commutativity/associativity), leaving only the single non-duplicated value in the final result.
- **Complexity: O(n) time, O(1) space** — strictly better on BOTH axes than the two obvious alternatives: a hash map counting occurrences (O(n) time, but O(n) extra space) or sorting first and scanning for the element without an adjacent duplicate (O(n log n) time). This is why it's worth knowing this specific trick by name rather than re-deriving a slower approach under time pressure.
- **A related but largely academic trick, worth knowing exists but not using:** XOR swap (`a ^= b; b ^= a; a ^= b;`) swaps two variables without a temporary. It silently breaks if `a` and `b` are the SAME memory location (XOR-ing a variable with itself zeroes it), and offers no real performance advantage over `std::swap` on modern hardware — know it exists as an XOR-properties exercise, don't reach for it in real code.

### Common Pitfalls
- Applying this trick to a variant where elements can appear an EVEN number of times other than exactly two (e.g. "every element appears three times except one") — the cancellation argument specifically relies on PAIRS canceling; a different multiplicity needs a different technique entirely (commonly, examining each bit position's count modulo the multiplicity, a related but distinct trick).


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// THE POWER OF XOR: every other element appears TWICE except one -- find that one,
// in O(n) time and O(1) space, using only XOR's algebraic properties.
//
// The properties that make this work:
//   x ^ x == 0        (a value XORed with itself cancels to zero)
//   x ^ 0 == x         (XOR with zero is the identity -- leaves x unchanged)
//   XOR is commutative and associative (order doesn't matter, grouping doesn't matter)
//
// XOR-ing every element together: all the DUPLICATED values cancel each other out
// (each pair XORs to 0), leaving only the single non-duplicate value.
int singleNumber(vector<int>& nums) {
    int result = 0;
    for (int num : nums) {
        result ^= num;
    }
    return result;
}

int main() {
    vector<int> nums{4, 1, 2, 1, 2};
    cout << "single number: " << singleNumber(nums) << "\n";   // expected 4

    vector<int> nums2{7};
    cout << "single number: " << singleNumber(nums2) << "\n";  // expected 7

    // COMPARE TO THE OBVIOUS APPROACHES: a hash map counting occurrences is O(n) time
    // but O(n) EXTRA space. Sorting first and scanning for the non-adjacent-duplicate is
    // O(n log n) time. XOR gets O(n) time AND O(1) space -- strictly better on both axes
    // than either alternative, which is why this specific trick is worth knowing by name
    // rather than re-deriving it from scratch under time pressure.

    // XOR SWAP (a related but largely academic trick): a ^= b; b ^= a; a ^= b; swaps two
    // variables without a temporary. Interesting as an XOR-properties exercise, but NOT
    // recommended in real code -- it silently breaks if a and b are the SAME memory
    // location (XORing a variable with itself zeroes it out), and offers no real
    // performance benefit over std::swap on modern hardware. Know it exists; don't use it.

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Counting Set Bits — Brian Kernighan's Algorithm

### The Why
The naive approach to counting set bits costs the same regardless of how many bits are actually set — checking all 32 (or 64) positions whether the number is `1` or `0xFFFFFFFF`. Brian Kernighan's trick instead does work PROPORTIONAL to the number of set bits specifically, which matters when working with sparse bitmasks.

### Core Mechanism
- **Naive approach:** check the lowest bit, shift right, repeat — O(number of bits in the representation), regardless of how many of them are actually 1.
- **Brian Kernighan's trick:** repeatedly apply `n = n & (n - 1)`, counting how many times this can be done before `n` reaches 0. Each application clears exactly the LOWEST set bit — so the loop runs exactly once per set bit, not once per bit position.
- **Why `n & (n-1)` clears exactly the lowest set bit:** subtracting 1 from `n` flips the lowest set bit to 0 AND flips every 0 bit below it to 1 (borrow propagation — the same mechanism as decimal subtraction, e.g. `1000 - 1 = 0111`). ANDing the original `n` with this `(n-1)`: everything below the lowest set bit was 0 in `n` (stays 0 regardless of what `n-1` has there), the lowest set bit itself is 1 in `n` but 0 in `n-1` (so it clears), and everything ABOVE the lowest set bit is identical in both `n` and `n-1` (so it's preserved unchanged). Net effect: exactly the lowest set bit disappears.
- The measured cross-check (naive vs. Kernighan agreeing on every test value) is a good general habit whenever implementing a "clever" version of something — verify it against the obvious version first, on values simple enough to reason about by hand if needed.

### Common Pitfalls
- Assuming Brian Kernighan's algorithm changes the WORST-case complexity (a number with all bits set, like `0xFFFFFFFF`, still takes just as many iterations as the naive approach) — its advantage is specifically for numbers with FEW set bits relative to their total bit width, not a universal improvement.


In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

// COUNTING SET BITS: the naive approach checks all 32 (or 64) bit positions one at a
// time, regardless of how many are actually set. Brian Kernighan's trick instead does
// exactly ONE iteration PER SET BIT -- much faster for numbers with few set bits.

int countBitsNaive(int n) {
    int count = 0;
    while (n) {
        count += n & 1;   // check the lowest bit
        n >>= 1;           // shift everything down by one
    }
    return count;
    // O(number of bits in the representation), REGARDLESS of how many are actually 1 --
    // a number with just one set bit still costs the same as a number with all bits set
}

int countBitsKernighan(int n) {
    int count = 0;
    while (n) {
        n = n & (n - 1);   // clears the LOWEST set bit -- see explanation below
        count++;
    }
    return count;
    // O(number of SET bits) -- a sparse number (few 1s) finishes almost immediately
}

int main() {
    for (int n : {0, 1, 7, 8, 255, 256, 1023}) {
        cout << "n=" << n << " naive=" << countBitsNaive(n)
             << " kernighan=" << countBitsKernighan(n) << "\n";
    }
    // both should always agree -- this cross-check is worth doing whenever implementing
    // a "clever" version of something: verify it against the obvious version first

    // WHY n & (n-1) CLEARS THE LOWEST SET BIT: subtracting 1 from n flips the lowest set
    // bit to 0 AND flips every 0 bit below it to 1 (borrow propagation, exactly like
    // decimal subtraction: 1000 - 1 = 0111). ANDing the original n with this (n-1) keeps
    // only the bits where BOTH agree -- since everything below the lowest set bit was
    // flipped to 1 in (n-1) but was already 0 in n, those stay 0; the lowest set bit
    // itself is 1 in n but 0 in (n-1), so it clears; everything ABOVE the lowest set bit
    // is unchanged in both, so it's preserved. Net effect: exactly the lowest set bit
    // disappears, nothing else changes.

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Power of Two and Isolating the Lowest Set Bit

### The Why
Both of these reuse the SAME underlying `n & (n-1)` family of reasoning from Brian Kernighan's algorithm — worth grouping together specifically to reinforce that a small number of core identities (lowest-set-bit manipulation, in this case) get reused across what look like unrelated problems.

### Core Mechanism
- **Power of two check:** a power of two has EXACTLY ONE set bit (1, 10, 100, 1000, ... in binary). Since `n & (n-1)` clears the lowest set bit, a number with only ONE set bit becomes exactly 0 after this operation — so `n > 0 && (n & (n-1)) == 0` correctly identifies powers of two. The `n > 0` guard is not optional: `0` has zero set bits (not a power of two), yet `0 & (0-1)` also happens to evaluate to 0 due to how integer underflow behaves, so without the explicit guard, 0 would be incorrectly reported as a power of two.
- **Isolating the lowest set bit:** `n & (-n)` extracts JUST the lowest set bit, zeroing out everything else. In two's complement representation, `-n` equals `~n + 1` — flipping all bits of `n` turns every 0 below the lowest set bit into 1 and the lowest set bit itself into 0; adding 1 then ripples through those new 1s (turning them back to 0) and sets the former lowest-set-bit position back to 1. Net effect: `-n` has all the SAME bits as `n` above the lowest set bit, but flipped; ANDing `n` with `-n` leaves only the one position where both agree — the lowest set bit itself.
- This `n & (-n)` isolation trick is exactly what powers the Fenwick Tree operations from the course README's Advanced Trees section — worth recognizing the connection if you're revisiting that material.

### Common Pitfalls
- Forgetting the `n > 0` guard on the power-of-two check — this is a real, easy-to-miss edge case, not a theoretical one, since `0`'s behavior under `n & (n-1)` coincidentally matches what a true power of two would produce.


In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

// A power of two has EXACTLY ONE set bit (1, 10, 100, 1000, ...). Combined with the
// n & (n-1) trick from Brian Kernighan's algorithm (clears the lowest set bit): if a
// number has only one set bit, clearing it leaves EXACTLY ZERO.
bool isPowerOfTwo(int n) {
    return n > 0 && (n & (n - 1)) == 0;
    // n > 0 guards against n=0 (which has zero set bits, and 0 & (0-1) happens to also
    // equal 0 due to integer underflow behavior, but 0 is NOT a power of two -- this
    // guard is not optional)
}

// isolate the LOWEST set bit of n (used in some Fenwick Tree operations, per the README's
// Advanced Trees section, and various bitmask algorithms)
int lowestSetBit(int n) {
    return n & (-n);
    // -n in two's complement is (~n + 1) -- flip all bits, add 1. Flipping all bits turns
    // every 0 below the lowest set bit into 1, and the lowest set bit itself into 0; adding
    // 1 then ripples through those new 1s, turning them back to 0 and setting the (former)
    // lowest set bit back to 1 -- everything ABOVE the lowest set bit ends up flipped
    // relative to n. ANDing n with this result: only the lowest set bit position has a 1
    // in BOTH n and -n, so that's the only bit that survives.
}

int main() {
    for (int n : {1, 2, 3, 4, 16, 17, 1024}) {
        cout << "isPowerOfTwo(" << n << ") = " << isPowerOfTwo(n) << "\n";
    }

    for (int n : {12, 40, 1}) {
        cout << "lowestSetBit(" << n << ") = " << lowestSetBit(n) << "\n";
    }
    // 12 = 1100 -> lowest set bit is 100 = 4
    // 40 = 101000 -> lowest set bit is 1000 = 8
    // 1 = 1 -> lowest set bit is 1

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Bitmasking for Subsets

### The Why
This is a representation technique as much as an algorithmic one: encoding a subset as a single integer, where bit `i` means "element `i` is included." It's directly comparable to `03_Backtracking`'s recursive include/exclude subset generation — same subsets produced, different mechanism — and it's the foundation for bitmask dynamic programming, a technique from a later phase of this course.

### Core Mechanism
- For a set of `n` elements, every integer from `0` to `2ⁿ - 1`, read in binary, IS a valid encoding of one possible subset — bit `i` set means element `i` is included, bit `i` clear means it's excluded.
- Iterate through all `2ⁿ` possible mask values; for each one, check every bit position to decode which elements that specific mask represents.
- **Complexity: O(2ⁿ · n)** — `2ⁿ` masks, O(n) work to decode each one into an actual subset list. This is the SAME complexity class as `03_Backtracking`'s recursive version (O(2ⁿ) calls total, amortized O(1) work per element across the whole recursion) — the bitmask version isn't asymptotically faster.
- **What this representation buys you instead of speed:** it's iterative (no recursion or call-stack overhead), and critically, each subset is DIRECTLY ADDRESSABLE by its mask value as an integer — which some problems specifically need, most notably bitmask dynamic programming, where a DP table is indexed BY the mask itself (`dp[mask] = ...`). That technique is a preview of a later phase of this course; the representation introduced here is the prerequisite for it.

### Common Pitfalls
- Reaching for bitmask subset generation on a set with more than roughly 20-25 elements — `2ⁿ` grows explosively (`2^25` is over 33 million), and this technique is only practical for genuinely small `n`, the same practical ceiling that applies to `03_Backtracking`'s recursive subset generation.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// BITMASKING FOR SUBSETS: represent EVERY possible subset of an n-element set as an
// n-bit integer, where bit i being set means "element i is included." This is an
// ALTERNATIVE to the recursive include/exclude backtracking from 03_Backtracking --
// same set of subsets produced, but iterative instead of recursive, and the "which
// elements are in this subset" question becomes direct bit-checking instead of a path
// built up through recursive calls.
vector<vector<int>> generateSubsetsBitmask(vector<int>& nums) {
    int n = (int)nums.size();
    vector<vector<int>> result;

    // every integer from 0 to 2^n - 1, in binary, IS a valid subset encoding --
    // iterate through all of them
    for (int mask = 0; mask < (1 << n); mask++) {
        vector<int> subset;
        for (int i = 0; i < n; i++) {
            if (mask & (1 << i)) {   // is bit i set in this mask?
                subset.push_back(nums[i]);
            }
        }
        result.push_back(subset);
    }
    return result;
}

int main() {
    vector<int> nums{1, 2, 3};
    auto subsets = generateSubsetsBitmask(nums);

    for (auto &s : subsets) {
        cout << "{ ";
        for (int x : s) cout << x << " ";
        cout << "}\n";
    }
    cout << "total=" << subsets.size() << " (2^3 = 8)\n";

    // COMPLEXITY: O(2^n * n) -- 2^n masks, O(n) work to decode each one into an actual
    // subset. Same overall complexity class as the recursive backtracking version from
    // 03_Backtracking (which is O(2^n) calls, each doing O(1) amortized work per element
    // across the whole recursion) -- the bitmask version isn't asymptotically FASTER, but
    // it's iterative (no recursion/call-stack overhead) and each subset is directly
    // addressable by its mask value, which some problems need (e.g. bitmask DP, where
    // you index a DP table BY the mask itself -- a preview of a technique from a later
    // phase, worth knowing this representation exists for when that arrives)

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. When Bit Tricks Are (and Aren't) Worth It

### The Why
Bit tricks have a genuine reputation problem: they're a favorite interview topic partly because they're compact and clever, which can tempt people into reaching for them even when a plainer approach would be equally fast and meaningfully more readable. This section is about calibrating that instinct.

### Core Mechanism
- **Bit tricks are worth it when:** the problem has a genuine space or time constraint that a bit-based approach specifically resolves (XOR's O(1) space beating a hash set's O(n), as in Section 3), or when the problem IS fundamentally about bits (counting set bits, checking specific bit properties, subset enumeration via bitmask DP later in the course).
- **Bit tricks are NOT automatically worth it when:** a clearer approach (a hash set, a simple loop, `std::bitset` from `03_STL_Deep_Dive`) achieves the same complexity class with meaningfully more readable code. `n & (n-1) == 0` is compact, but `popcount(n) == 1` (using a set-bit-counting function) communicates the same check with less required background knowledge from whoever reads it later — including future you.
- **A fast self-check:** if you can't explain WHY a bit trick works in one or two sentences (the way each technique in this notebook was justified, not just stated), that's a signal you're pattern-matching rather than understanding it — and code you don't fully understand is a liability regardless of how fast it runs.
- In real production code (as opposed to competitive programming, where every microsecond and every byte can matter), prefer clarity unless profiling has specifically identified a bit-manipulation bottleneck — the same "prefer the STL over a hand-rolled version" guidance from `03_STL_Deep_Dive` and `04_Sorting_Algorithms` applies here too: know these tricks well enough to recognize and use them when they're the right tool, but don't reach for them reflexively.

### Common Pitfalls
- Using a bit trick specifically to look clever in an interview or code review, without being able to justify it beyond "it's faster" or "I've seen this before" — an interviewer (or a future maintainer) asking "why does this work" deserves a real answer, not just correct output.


## 8. Phase Checkpoint, Cheat Sheet, and Answer Key

### Bit Trick Cheat Sheet

| Operation | Formula | Why it works |
|---|---|---|
| Check bit i | `(n >> i) & 1` | Shift target bit to position 0, mask with 1 |
| Set bit i | `n \| (1 << i)` | OR's identity (0) preserves other bits; 1 forces the target on |
| Clear bit i | `n & ~(1 << i)` | AND's identity (1) preserves other bits; 0 forces the target off |
| Toggle bit i | `n ^ (1 << i)` | XOR's identity (0) preserves other bits; 1 flips the target |
| Clear lowest set bit | `n & (n - 1)` | Borrow propagation in `n-1` cancels exactly the lowest set bit under AND |
| Isolate lowest set bit | `n & (-n)` | Two's complement negation flips everything above the lowest set bit |
| Check power of two | `n > 0 && (n & (n-1)) == 0` | A power of two has exactly one set bit, which clearing removes entirely |
| Find the single non-duplicate | XOR all elements | Paired duplicates cancel (`x^x=0`); identity (`x^0=x`) leaves the singleton |
| Enumerate all subsets | Iterate `mask` from `0` to `2ⁿ-1` | Every integer's binary representation IS a valid subset encoding |

### Checkpoint Questions
1. Why does the "clear bit" operation use `n & ~(1 << i)` instead of, say, `n | (1 << i)`?
2. Explain, from first principles, why XOR-ing an entire array (where every value appears exactly twice except one) isolates the single non-duplicate.
3. Trace `n & (n-1)` by hand for `n = 12` (binary `1100`). What's the result, and why?
4. Why does `n & (n-1) == 0` correctly identify powers of two, and why is the `n > 0` guard necessary?
5. What's the actual advantage of bitmask subset representation over `03_Backtracking`'s recursive version, given they're the same complexity class?
6. Give an example of when using a bit trick would be a WORSE choice than a plainer alternative, even if both are equally fast.

### Answer Key
1. OR can only ever turn bits ON, never off — using it to "clear" a bit would be structurally wrong regardless of the mask. Clearing requires AND (which can turn bits off) combined with a mask that's 1 everywhere except the target position — that's exactly what `~(1 << i)` provides.
2. XOR has two relevant properties: `x ^ x == 0` (a value XORed with itself cancels) and `x ^ 0 == x` (XOR with zero is the identity). XOR is also commutative and associative, so grouping and order don't matter. XOR-ing the whole array together, every DUPLICATED value's two occurrences cancel to 0 (regardless of where they appear relative to each other), leaving only the singleton value XORed with a running total of zeros — which equals the singleton itself.
3. `n - 1 = 11` (binary `1011`). `n & (n-1) = 1100 & 1011 = 1000` (decimal 8). The lowest set bit of `1100` (bit position 2, value 4) gets cleared, leaving `1000`.
4. A power of two has exactly one set bit. `n & (n-1)` clears the lowest set bit — if there's only ONE set bit total, clearing it leaves exactly 0, so `(n & (n-1)) == 0` is true precisely when there's at most one set bit. The `n > 0` guard is necessary because `n = 0` also produces `(0 & -1) == 0` due to how integer representation behaves, but 0 has ZERO set bits and is not a power of two — without the guard, this edge case is silently misclassified.
5. Direct addressability: each subset corresponds to a specific integer mask value, which can be used directly as an array/table index. This matters specifically for bitmask dynamic programming (a later-phase technique), where you need `dp[mask]` to look up or store a result for a specific subset — something a recursive call stack doesn't naturally give you without extra bookkeeping.
6. Using `n & (n-1) == 0` to check for a power of two in code a broader team maintains, when a more explicit function (or even a named helper like `isPowerOfTwo`) with a comment communicates the same check without requiring every future reader to already know this specific bit identity. Both run equally fast; only one is immediately clear to someone unfamiliar with the trick.

### Mini Project
Implement `singleNumberII(nums)`: every element in the array appears exactly THREE times except one, which appears exactly once — find that element.
1. Notice Section 3's plain XOR trick doesn't work here directly — triples don't cancel to 0 under XOR the way pairs do.
2. Research (or derive) the bit-counting variant: for each of the 32 bit positions, count how many numbers in the array have that bit set. If a bit's count is NOT divisible by 3, that bit must belong to the singleton (since every OTHER number contributes that bit exactly 3 times, in multiples of 3, while the singleton contributes it 0 or 1 extra time).
3. Reconstruct the singleton value bit by bit from these counts, and verify against a brute-force hash-map-counting approach.

This exercises: recognizing when a bit trick's underlying assumption (pairs specifically) doesn't generalize to a superficially similar problem (triples), and adapting the reasoning rather than the formula — the same "understand why, not just what" instinct this whole notebook has been building toward.


## 9. LeetCode Practice Problems

Grouped by pattern. Hints point at the pattern's key decision, not the full solution.

### Basic Bit Operations

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 191 | Number of 1 Bits | Easy | Section 4, directly (either naive or Brian Kernighan's works; try both) |
| 231 | Power of Two | Easy | Section 5, directly |
| 342 | Power of Four | Easy | Power-of-two check from Section 5, PLUS an additional check on which bit position is set (powers of four only have their single set bit at even-numbered positions) |
| 190 | Reverse Bits | Easy | Build the result bit by bit, reading the input from one end and writing to the other |

### XOR Tricks

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 136 | Single Number | Easy | Section 3, directly |
| 137 | Single Number II | Medium | This notebook's Mini Project — the bit-counting-modulo-3 variant |
| 260 | Single Number III | Medium | TWO singletons this time (everything else still paired) — XOR everything first to get `a^b`, then use a set bit in that result to partition the array into two groups, each containing exactly one singleton |
| 268 | Missing Number | Easy | XOR every array value with every index from 0 to n — the "missing" number is what survives, using the same pairing-cancellation idea as Section 3 |

### Bitmasking

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 78 | Subsets | Medium | Section 6, directly — compare your bitmask solution against the recursive version from `03_Backtracking` |
| 338 | Counting Bits | Easy | Compute set-bit counts for every number from 0 to n; look for a way to reuse a previously computed answer (`dp[i] = dp[i >> 1] + (i & 1)`) instead of recomputing each one from scratch |
| 1178 | Number of Valid Words for Each Puzzle | Hard | Represent each word and puzzle as a 26-bit mask (one bit per letter present) — turns substring/letter-membership checks into O(1) bitmask comparisons |

### Self-Check Before Moving to `06_String_Algorithms`
If you can explain each cheat-sheet formula's reasoning without looking at it — not just recite the formula — you've gotten the actual value this notebook offers. Bit tricks are narrow in scope compared to the patterns in earlier notebooks, but the ones covered here (especially XOR-cancellation and bitmask subset representation) reappear as small building blocks inside larger problems throughout the rest of this course, including in `06_String_Algorithms`, where character-set membership is a natural fit for the same bitmask idea from Section 6.
